# Magnetic Resonance Imaging II
 
## Exercise 5
 
---
 
**Prof. Dr. Markus Zimmermann**
 
**Mohamed Wassim Bensalem**
 
---

In Exercise 4 you solved the radial Compressed Sensing reconstruction problem
 
$$\min_{\mathbf{x}} \;\frac{1}{2} \|\mathbf{A}\mathbf{x} - \mathbf{d}\|_2^2 + \lambda \|\mathbf{G}\mathbf{x}\|_{2,1}$$
 
by unrolling ADMM and using isotropic Total Variation as the regularizer $R(\mathbf{x}) = \|\mathbf{G}\mathbf{x}\|_{2,1}$. TV works well, but it is a fixed, hand-designed prior: it only ever encodes the assumption "images are piecewise smooth." It does not know anything about the specific anatomy in this scan, and its strength is controlled by a single scalar $\lambda$ that you had to tune by hand.
 
This exercise replaces the hand-crafted TV proximal step with a **learned proximal operator** — a small convolutional neural network that plays the same role as `joint_soft_thresholding`, but whose behaviour is *learned from data* rather than derived analytically. The surrounding ADMM structure — the CG data-consistency step, the scaled dual variable $\mathbf{u}$, the outer iteration loop — stays exactly the same. Only the $\mathbf{z}$-update changes:
 
$$\mathbf{z} \leftarrow \text{prox}_{\lambda/\rho}(\mathbf{G}\mathbf{x} + \mathbf{u}) \qquad \longrightarrow \qquad \mathbf{z} \leftarrow f_\theta(\mathbf{x} + \mathbf{u})$$
 
where $f_\theta$ is a ResNet with learnable weights $\theta$. Because the whole unrolled ADMM loop is now built from differentiable PyTorch operations, you can backpropagate through it — not just through $f_\theta$, but through the CG iterations themselves — and train both the network weights $\theta$ and the scalar $\rho$ jointly, end to end.
 
The catch: you don't have a training set of paired (undersampled, fully-sampled) images, and you won't be given one. Instead you will train **zero-shot and self-supervised**, directly on the single scan you already acquired. The trick (following the SSDU / self-supervised k-space splitting idea) is to repeatedly split the acquired k-space samples into two random halves: reconstruct from one half, and check whether the result is consistent with the *other, unseen* half. No ground-truth image ever enters the loss — the network only ever learns from the internal consistency of the measurement itself.
 
By the end of this exercise you will have four reconstructions of the same undersampled data to compare side by side: CG (unregularized), classical ADMM with fixed TV, DeepADMM with a randomly initialized (untrained) network, and DeepADMM after self-supervised training.

## Software Imports
 
All required libraries and modules are imported here.

In [ ]:
%matplotlib widget

import sys
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch
import torch.nn as nn
from tqdm import tqdm

sys.path.insert(0, "/home/jovyan/mri/src")
sys.path.insert(1, "/home/jovyan/shared-folder/Magnetresonanztomographie-2/mri/src")

from mri.conopt.functional.fourier import (
    nonuniform_fourier_transform_adjoint,
    nonuniform_fourier_transform_forward,
)
from mri.conopt.linops import LinearOperator
from mri.conopt.linops.composite import ConcatOperator
from mri.model.regularizer import Regularizer
from mri.parim.analytic import cyclic_head_coil

## System and Sequence Parameters
 
`R` controls the undersampling factor: the sequence acquires `n_y // R` radial spokes instead of the full `n_y`. Start with `R = 1` to verify the full pipeline, then increase to `R = 4` to observe the streaking artifacts and see how CS suppresses them. `num_dummy_pulses` sets the number of preparatory TRs played before data acquisition begins — see the Sequence Components section for details.

In [ ]:
seq_filename = "gre_radial.seq"

fov_x, fov_y = 256e-3, 256e-3
n_x, n_y = 128, 128
n_channels = 8
R = 4
num_dummy_pulses = 10

case = ["T1-weighted", "T2-weighted", "PD-weighted"][0]

if case == "T1-weighted":
    flip_angle_deg = 90
    tr = 250e-3
    te = 5e-3
elif case == "T2-weighted":
    flip_angle_deg = 40
    tr = 5000e-3
    te = 50e-3
elif case == "PD-weighted":
    flip_angle_deg = 40
    tr = 5000e-3
    te = 5e-3

torch.set_default_device("cpu")

## System and Sequence Object

In [ ]:
system = pp.Opts(
    max_grad=28,
    grad_unit="mT/m",
    max_slew=150,
    slew_unit="T/m/s",
)

seq = pp.Sequence(system)

## Linear Operators

`ChannelOperator`, `NonuniformFourierTransformOperator`, and `GradientOperator` are identical to Exercise 4 and are provided here so you can focus on the new material. One operator is new: `IdentityOperator`. In classical ADMM, the $\mathbf{x}$-update solves $(\mathbf{Q} + \rho\,\mathbf{G}^\mathrm{H}\mathbf{G})\,\mathbf{x} = \dots$, using the gradient operator $\mathbf{G}$ to couple $\mathbf{x}$ to the TV proximal step. DeepADMM's proximal step no longer acts on $\mathbf{G}\mathbf{x}$ — the network $f_\theta$ acts directly on the image $\mathbf{x}$ itself — so the coupling term in the $\mathbf{x}$-update simplifies to $\rho\,\mathbf{I}$ instead of $\rho\,\mathbf{G}^\mathrm{H}\mathbf{G}$. `IdentityOperator` provides that $\mathbf{I}$.

In [ ]:
class ChannelOperator(LinearOperator):

    def __init__(
        self,
        coil_sensitivities: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        nC = coil_sensitivities.shape[0]

        self.coil_sensitivities = coil_sensitivities

        self.forward_shape = (1, *data_shape)
        self.adjoint_shape = (nC, *data_shape)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = self.coil_sensitivities * x
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = torch.sum(torch.conj(self.coil_sensitivities) * x, dim=0, keepdim=True)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)


class NonuniformFourierTransformOperator(LinearOperator):

    def __init__(
        self,
        k: torch.Tensor,
        input_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        nD, nK = k.shape
        nC, nX, nY = input_shape
        self.n_modes = (nX, nY)

        self.k = k
        self.forward_shape = (nC, nX, nY, 1)
        self.adjoint_shape = (nC, nK)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = nonuniform_fourier_transform_forward(self.k, x)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = nonuniform_fourier_transform_adjoint(self.k, x, self.n_modes)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)


class GradientOperator(LinearOperator):

    def __init__(
        self,
        dim: int,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        self.dim = dim
        self.forward_shape = data_shape
        self.adjoint_shape = data_shape
        self.shape = (
            int(2 * torch.prod(torch.tensor(data_shape))),
            int(2 * torch.prod(torch.tensor(data_shape))),
        )

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = x - torch.roll(x, 1, self.dim)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = x - torch.roll(x, -1, self.dim)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)


class IdentityOperator(LinearOperator):

    def __init__(
        self,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):

        self.shape = (
            int(2 * torch.prod(torch.tensor(data_shape))),
            int(2 * torch.prod(torch.tensor(data_shape))),
        )

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        return x

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        return x

## Numerical Solvers

`conjugate_gradient`, `joint_soft_thresholding`, and `admm` are identical to Exercise 4 and are provided here unchanged. You will use `conjugate_gradient` again inside DeepADMM's $\mathbf{x}$-update; `joint_soft_thresholding` and `admm` are kept so you can run the classical fixed-TV reconstruction as a baseline for comparison later in this notebook.

In [ ]:
def joint_soft_thresholding(
    g: torch.Tensor,
    lamda: float,
    num_gradient_dims: int = 2,
) -> torch.Tensor:

    g = torch.reshape(g, (num_gradient_dims, -1))

    r = torch.sqrt(torch.sum(g**2, axis=0))

    r = torch.fmax(r, torch.tensor((lamda,)))

    factor = (r - lamda) / r

    g = factor * g

    return g.ravel()


def admm(
    Q: LinearOperator,
    b: torch.Tensor,
    G: LinearOperator,
    x: torch.Tensor,
    lamda: float,
    rho: float,
    num_gradient_dims: int = 2,
    maxiter_admm: int = 30,
    maxiter_cg: int = 5,
) -> torch.Tensor:

    z = torch.zeros(G.shape[0], dtype=x.dtype)
    u = torch.zeros(G.shape[0], dtype=x.dtype)

    for iteration in tqdm(range(maxiter_admm)):

        x = conjugate_gradient(
            Q + rho * G.H @ G,
            b=b + rho * G.H @ (z - u),
            x=x,
            maxiter=maxiter_cg,
            disable_progressbar=True,
        )

        z = joint_soft_thresholding(
            G @ x + u,
            lamda / rho,
            num_gradient_dims,
        )

        u = u + G @ x - z

    return x


def conjugate_gradient(
    Q: LinearOperator,
    b: torch.Tensor,
    x: torch.Tensor,
    maxiter: int = 10,
    disable_progressbar: bool = False,
) -> torch.Tensor:

    g_prev = torch.ones(x.shape, dtype=x.dtype)
    p_prev = torch.zeros(x.shape, dtype=x.dtype)

    for iteration in tqdm(
        range(maxiter),
        disable=disable_progressbar,
    ):

        g = Q @ x - b

        conjugacy_factor = g @ g / (g_prev @ g_prev)
        p = g + conjugacy_factor * p_prev
        stepsize = g @ g / (p @ (Q @ p))
        x = x - stepsize * p

        g_prev = g.clone()
        p_prev = p.clone()

    return x

## Sequence Components
 
The radial GRE sequence, its timing calculations, and the (undersampled) spoke assembly below are identical to Exercise 4 and are provided here so you can focus on the new reconstruction material. As a reminder: there are no fixed `gx`/`gy` gradients — a single readout gradient `g_ro` and pre-phaser `g_pre` are defined along x, and their areas are projected onto x and y for each spoke angle in the sequence loop below. `R` controls the undersampling factor: the sequence acquires `n_y // R` radial spokes instead of the full `n_y`.

In [ ]:
delta_kx = 1 / fov_x
raster_eps = 1e-12

rf = pp.make_sinc_pulse(
    flip_angle=np.deg2rad(flip_angle_deg),
    system=system,
    use="excitation",
)

g_ro = pp.make_trapezoid(
    channel="x",
    flat_area=n_x * delta_kx,
    flat_time=3.2e-3,
    system=system,
)

g_pre = pp.make_trapezoid(
    channel="x",
    area=-g_ro.area / 2,
    system=system,
)

adc = pp.make_adc(
    num_samples=n_x,
    duration=g_ro.flat_time,
    delay=g_ro.rise_time,
    system=system,
)

gx_spoiler = pp.make_trapezoid(
    channel="x",
    area=4 * g_ro.area,
    system=system,
)

gy_spoiler = pp.make_trapezoid(
    channel="y",
    area=4 * g_ro.area,
    system=system,
)

te_delay = (
    te
    - (pp.calc_duration(rf) - pp.calc_rf_center(rf)[0])
    - pp.calc_duration(g_pre)
    - pp.calc_duration(g_ro) / 2
)

tr_delay = (
    tr
    - pp.calc_duration(rf)
    - pp.calc_duration(g_pre)
    - pp.calc_duration(g_ro)
    - pp.calc_duration(gx_spoiler)
    - te_delay
)

te_delay = (
    np.ceil((te_delay - raster_eps) / seq.grad_raster_time) * seq.grad_raster_time
)
tr_delay = (
    np.ceil((tr_delay - raster_eps) / seq.grad_raster_time) * seq.grad_raster_time
)

assert np.all(te_delay >= 0)
assert np.all(tr_delay >= 0)

## (Undersampled) Radial Sequence Assembly

Provided unchanged from Exercise 4. Each TR acquires one spoke through the center of k-space at angle $\theta_i = i\pi / (n_y / R)$, for $i = 0, \dots, n_y/R - 1$; `num_dummy_pulses` preparatory TRs (ADC off) are played first to drive the magnetization into steady state.

In [ ]:
angles = np.linspace(0, np.pi, num=n_y // R, endpoint=False)

for index in range(-num_dummy_pulses, n_y // R):

    if index < 0:
        angle = 0
    else:
        angle = angles[index]

    seq.add_block(rf)

    gx_pre = pp.make_trapezoid(
        channel="x",
        area=np.cos(angle) * g_pre.area,
        duration=pp.calc_duration(g_pre),
        system=system,
    )

    gy_pre = pp.make_trapezoid(
        channel="y",
        area=np.sin(angle) * g_pre.area,
        duration=pp.calc_duration(g_pre),
        system=system,
    )

    gx_ro = pp.make_trapezoid(
        channel="x",
        area=np.cos(angle) * g_ro.area,
        duration=pp.calc_duration(g_ro),
        system=system,
    )

    gy_ro = pp.make_trapezoid(
        channel="y",
        area=np.sin(angle) * g_ro.area,
        duration=pp.calc_duration(g_ro),
        system=system,
    )

    seq.add_block(gx_pre, gy_pre)

    seq.add_block(pp.make_delay(te_delay))

    if index < 0:
        seq.add_block(gx_ro, gy_ro)
    else:
        seq.add_block(gx_ro, gy_ro, adc)

    seq.add_block(gx_spoiler, gy_spoiler)

    seq.add_block(pp.make_delay(tr_delay))

## Verification and Export

In [ ]:
ok, error_report = seq.check_timing()

if ok:
    print("Timing check passed successfully")
else:
    print("Timing check failed. Error listing follows:")
    [print(e) for e in error_report]

plt.figure()
seq.plot(time_range=(0.0, tr - tr_delay + te_delay))

seq.write(seq_filename, v141_compat=True)

## Phantom and Multi-Channel Simulation

Run this cell as-is.

In [ ]:
phantom = mr0.util.load_phantom(size=(n_x, n_y))

coil_sensitivities_recon = cyclic_head_coil(
    (n_channels, n_x, n_y, 1),
).to(torch.complex64)
coil_sensitivities_recon = coil_sensitivities_recon.flip((1, 2))
coil_sensitivities_recon = coil_sensitivities_recon[..., 0]

coil_sensitivities_sim = cyclic_head_coil(
    (n_channels, n_x, n_y, 1),
    fov_scaling=(phantom.size[0] / fov_x, phantom.size[1] / fov_y, 1),
).to(torch.complex64)
phantom.coil_sens = coil_sensitivities_sim

plt.figure()
phantom.plot()

mr0_seq = mr0.Sequence.import_file(seq_filename)

data = phantom.build()
graph = mr0.compute_graph(mr0_seq, data)
signal = mr0.execute_graph(graph, mr0_seq, data, print_progress=True)

## Coil Images

Provided unchanged from Exercise 4. `signal` is reshaped and permuted to `data` of shape $(n_\text{channels}, n_K)$; the k-space trajectory `k_traj_adc` is extracted with `seq.calculate_kspace()`, restricted to $k_x, k_y$, and normalized to $[-0.5, 0.5]$. `nonuniform_fourier_transform_adjoint` reconstructs each coil image from the non-Cartesian samples, since a standard inverse FFT cannot be used on radial data.

In [ ]:
data = signal.clone()
data = data.reshape((-1, n_channels)).permute((1, 0))
data = data.contiguous()

kspace = seq.calculate_kspace()
k_traj_adc = kspace[0]
k_traj_adc = k_traj_adc[:2, :]
k_traj_radius = np.sqrt(np.sum(k_traj_adc**2, axis=0, keepdims=True))
k_traj_adc = k_traj_adc / k_traj_radius.max() * 0.5
k_traj_adc = torch.tensor(k_traj_adc, dtype=torch.float32)

coil_images = nonuniform_fourier_transform_adjoint(k_traj_adc, data, (n_x, n_y))[..., 0]

plt.figure()
for i in range(n_channels):
    plt.subplot(2, 4, i + 1)
    plt.imshow(torch.abs(coil_images[i, :, :]).numpy().T, cmap="gray")
    plt.title(f"Channel {i + 1}")
plt.suptitle("Coil images")
plt.tight_layout()
plt.show()

plt.figure()
for i in range(n_channels):
    plt.subplot(2, 4, i + 1)
    plt.imshow(torch.abs(coil_sensitivities_recon[i, :, :]).numpy().T, cmap="viridis")
    plt.title(f"Channel {i + 1}")
plt.suptitle("Coil sensitivity maps")
plt.tight_layout()
plt.show()

## Compose the Forward Operator and Compute $\mathbf{A}^\mathrm{H} \mathbf{d}$
 
Provided unchanged from Exercise 4. The forward operator for radial parallel imaging is $\mathbf{A} = \mathbf{F}_\text{nu}\mathbf{C}$ — there is no mask operator here, since the trajectory `k_traj_adc` already encodes which k-space locations were sampled. `AHd` is the matched-filter (zero-filled, regridded) reconstruction and serves as the baseline for every comparison in this notebook.

In [ ]:
F = NonuniformFourierTransformOperator(k_traj_adc, (n_channels, n_x, n_y))
C = ChannelOperator(coil_sensitivities_recon, (n_x, n_y))

A = F @ C

AHd = A.H @ data.view(torch.float32).ravel()

AHd_image = AHd.view(torch.complex64).reshape((n_x, n_y))

plt.figure()
plt.imshow(torch.abs(AHd_image).numpy().T, cmap="gray")
plt.title(
    f"$\\mathbf{{A}}^\\mathrm{{H}} \\mathbf{{d}}$ — Regridded Reconstruction ({case})"
)
plt.show()

## Iterative Reconstruction with Conjugate Gradient
 
Provided unchanged from Exercise 4. `conjugate_gradient` solves $\mathbf{Q}\mathbf{x} = \mathbf{b}$ with $\mathbf{Q} = \mathbf{A}^\mathrm{H}\mathbf{A}$ and $\mathbf{b} = \mathbf{A}^\mathrm{H}\mathbf{d}$, giving the unregularized least-squares reconstruction. `Q` and `b` computed here are reused unchanged throughout the rest of the notebook — by the classical ADMM baseline, and later as the fixed operator passed into DeepADMM at inference time.

In [ ]:
Q = A.H @ A

b = AHd
x = torch.zeros(n_x * n_y, dtype=torch.complex64).view(torch.float32)

x_recon = conjugate_gradient(Q, b, x, maxiter=30)

x_recon_image = x_recon.view(torch.complex64).reshape((n_x, n_y))

## Visualization

In [ ]:
plt.figure()
plt.subplot(1, 2, 1)
plt.imshow(torch.abs(AHd_image).numpy().T, cmap="gray")
plt.title(f"Zero-filled ({case})")
plt.subplot(1, 2, 2)
plt.imshow(torch.abs(x_recon_image).numpy().T, cmap="gray")
plt.title(f"CG Reconstruction ({case})")
plt.tight_layout()
plt.show()

## Compressed Sensing Reconstruction with ADMM
 
Provided unchanged from Exercise 4. `Gx`/`Gy` are the finite-difference gradient operators along each spatial axis, stacked into a single joint gradient operator `G = ConcatOperator((Gx, Gy))`; `admm` then solves the TV-regularized problem $\min_{\mathbf{x}} \frac{1}{2}\|\mathbf{A}\mathbf{x} - \mathbf{d}\|_2^2 + \lambda\|\mathbf{G}\mathbf{x}\|_{2,1}$ with `lamda=5e-2`, using the fixed proximal step `joint_soft_thresholding` in place of a learned one. This is the classical baseline that DeepADMM is compared against throughout the rest of the notebook.

In [ ]:
Gx = GradientOperator(0, (n_x, n_y))
Gy = GradientOperator(1, (n_x, n_y))

G = ConcatOperator((Gx, Gy))

x_admm = admm(
    Q,
    b,
    G,
    b,
    lamda=5e-2,
    rho=1,
    maxiter_admm=100,
    maxiter_cg=5,
)

x_admm_image = x_admm.view(torch.complex64).reshape((n_x, n_y))

## Intermediate Checkpoint: Classical Reconstructions
 
Provided unchanged from Exercise 4. Before moving on to DeepADMM, this plot gives you a quick visual sanity check that the classical pipeline — zero-filled, CG, and fixed-TV ADMM — is behaving as expected at the current `R` and `lamda` before you start comparing against the learned reconstruction. The full discussion questions comparing *all* reconstructions, including DeepADMM, are in the final Comparison section at the end of the notebook.

In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(1, 3, 1)
plt.imshow(torch.abs(AHd_image).numpy().T, cmap="gray")
plt.title(f"Zero-filled ({case})")
plt.subplot(1, 3, 2)
plt.imshow(torch.abs(x_recon_image).numpy().T, cmap="gray")
plt.title(f"CG Reconstruction ({case})")
plt.subplot(1, 3, 3)
plt.imshow(torch.abs(x_admm_image).numpy().T, cmap="gray")
plt.title(f"Compressed Sensing ({case})")
plt.tight_layout()
plt.show()

## Task 1: Model
 
You will now implement **DeepADMM**, which unrolls the same three-step ADMM iteration as Exercise 4, but replaces the fixed TV proximal operator with a learned `Regularizer` module, and makes the penalty parameter $\rho$ a trainable weight instead of a fixed hyperparameter:
 
$$\mathbf{x} \leftarrow (\mathbf{Q} + \rho\, \mathbf{I})^{-1}(\mathbf{b} + \rho\, (\mathbf{z} - \mathbf{u}))$$
 
$$\mathbf{z} \leftarrow f_\theta(\mathbf{x} + \mathbf{u})$$
 
$$\mathbf{u} \leftarrow \mathbf{u} + \mathbf{x} - \mathbf{z}$$
 
Because $f_\theta$ acts directly on the image rather than on its gradient, the $\mathbf{x}$-update uses $\mathbf{I}$ (the `IdentityOperator`) in place of $\mathbf{G}^\mathrm{H}\mathbf{G}$ — this is why we needed the new operator above. Everything else has the same shape and role as in Exercise 4's `admm`: `Q`/`b` still come from the data-fidelity term, and `maxiter_cg` still controls how many CG steps are used per outer iteration (a small number, since ADMM only needs an approximate update at every step thanks to warm-starting).
 
**Your tasks:**
 
1. Subclass `nn.Module`. In `__init__`, register `self.rho` as an `nn.Parameter` (initialized from `rho_init`) so it is optimized alongside the network — this is what makes $\rho$ *learned* rather than *fixed*, unlike in Exercise 4's `admm`.
2. Instantiate `self.regularizer = Regularizer(shape)` and `self.I = IdentityOperator(shape)` as submodules/attributes in `__init__`.
3. In `forward(self, Q, b, x, disable_progressbar=False)`, initialize the dual variables `z` and `u` as zero tensors of shape `self.I.shape[0]`, matching `x`'s dtype and device.
4. Implement the loop: run `conjugate_gradient` with the coupling operator `Q + self.rho * self.I` and right-hand side `b + self.rho * (z - u)`, using `x` as the warm-start initial guess; then update `z` by calling `self.regularizer(x + u)`; then update `u` in place with the standard ADMM dual update.
5. Return the final `x` after `maxiter_admm` outer iterations.
Because `self.rho` is an `nn.Parameter` and `self.regularizer` is an `nn.Module`, both are registered automatically and will receive gradients when you call `.backward()` on a loss computed from `forward`'s output.

In [ ]:
# Your solution here




## Task 2: Inference on Non-Trained Model
 
Before training anything, run DeepADMM with **randomly initialized** weights on the full (unsplit) k-space data, using the same `Q`/`b` from the CG reconstruction above. This gives a fair baseline: it isolates what the unrolled ADMM *structure* contributes on its own, so any further improvement after training in Task 3 can be attributed to the learned weights.
 
DeepADMM only runs `maxiter_admm=10` outer iterations rather than to convergence, so — unlike CG — its initial guess matters. Initialize `x` from the regridded solution `b` rather than from zero, so the network only needs to refine an image that already has most of its low-frequency content, instead of building one from nothing in 10 steps.
 
**Your tasks:**
 
1. Fix a random seed with `torch.manual_seed(...)` so the untrained network's initialization is reproducible.
2. Instantiate `model_init = DeepADMM(shape, ...)` with `shape = (n_x, n_y, 1)` — the exercise data has no extra contrast dimensions, so a trailing `nZ = 1` is all that's needed.
3. Run `model_init(Q, b, b)` inside a `torch.no_grad()` block — `b`, the regridded solution, serves as both the data-consistency target and the initial image guess.
4. Add the result to the four-panel comparison plot below, alongside zero-filled, CG, and classical ADMM.

In [ ]:
# Your solution here




## Task 3: Training

You have no ground-truth image and no external training set — only the single radial k-space acquisition you already simulated. Training therefore has to be **zero-shot and self-supervised**: on every training iteration, randomly split the acquired k-space samples into two disjoint halves, reconstruct an image using only one half (`src`), and measure how well that reconstruction predicts the *other, held-out* half (`tar`) when mapped back into k-space. Because the split is random and different on every iteration, over many iterations the network is pushed to produce reconstructions that are consistent with the *entire* measured k-space, without ever seeing a fully-sampled reference image.
 
$$\mathcal{L} = \frac{\|\mathbf{F}_\text{tar}\mathbf{C}\,\hat{\mathbf{x}} - \mathbf{d}_\text{tar}\|_1}{\|\mathbf{d}_\text{tar}\|_1}, \qquad \hat{\mathbf{x}} = \text{DeepADMM}(\mathbf{Q}_\text{src}, \mathbf{b}_\text{src}, \mathbf{x}_0)$$
 
where $\mathbf{Q}_\text{src} = \mathbf{C}^\mathrm{H}\mathbf{F}_\text{src}^\mathrm{H}\mathbf{F}_\text{src}\mathbf{C}$ and $\mathbf{b}_\text{src} = \mathbf{C}^\mathrm{H}\mathbf{F}_\text{src}^\mathrm{H}\mathbf{d}_\text{src}$ are built only from the source half, exactly as $\mathbf{Q}$ and $\mathbf{b}$ were built from the full trajectory earlier in the notebook.
 
**Your tasks:**
 
1. Instantiate a fresh `model = DeepADMM(shape, ...)` and an `Adam` optimizer over `model.parameters()`.
2. Loop over `num_splits` iterations. On each iteration: shuffle the k-space sample indices and split them in half into `index_src`/`index_tar`; slice `k_traj_adc`/`data` accordingly to build `k_traj_src`, `k_data_src`, `k_traj_tar`, `k_data_tar`.
3. Build `F_src`, `F_tar`, and `C` from the split trajectories, then `Q_src` and `b_src` from `F_src`/`C` alone.
4. Run `model(Q_src, b_src, b_src)` to get the reconstruction — `b_src` again serves as both the data-fidelity term and the initial guess, as in Task 2.
5. Map the reconstruction back to k-space at the *target* trajectory with `F_tar @ C @ x_out`, and compute the normalized $\ell_1$ loss against `k_data_tar`.
6. Call `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()`; record the loss for the final plot and attach it to the progress bar (`progress_bar.set_postfix(...)`) so you can monitor training live rather than only after the fact.

There is no image-domain loss anywhere in this loop — only k-space consistency on samples the network never saw when producing its reconstruction. With just `num_splits` gradient steps (one per split, no epochs), don't expect a smooth loss curve; some noise between splits is normal, since each one draws a different random half of the data.

In [ ]:
shape = (n_x, n_y, 1)

torch.manual_seed(0)
model = DeepADMM(
    shape,
    rho_init=0.1,
    maxiter_admm=10,
    maxiter_cg=5,
)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)

nK = k_traj_adc.shape[1]
num_splits = 25

model.train()
losses = []

progress_bar = tqdm(range(num_splits), desc="Self-supervised splits")

for split in progress_bar:

    np.random.seed(split)

    indices = np.arange(nK)
    np.random.shuffle(indices)
    index_src, index_tar = np.split(indices, 2)

    k_traj_src = k_traj_adc[:, index_src]
    k_data_src = data[:, index_src].view(torch.float32).ravel()

    k_traj_tar = k_traj_adc[:, index_tar]
    k_data_tar = data[:, index_tar].view(torch.float32).ravel()

    F_src = NonuniformFourierTransformOperator(k_traj_src, (n_channels, n_x, n_y))
    F_tar = NonuniformFourierTransformOperator(k_traj_tar, (n_channels, n_x, n_y))
    C = ChannelOperator(coil_sensitivities_recon, (n_x, n_y))

    Q_src = C.H @ F_src.H @ F_src @ C
    b_src = C.H @ F_src.H @ k_data_src

    x_out = model(Q_src, b_src, b_src, disable_progressbar=True)

    k_data_pred = F_tar @ C @ x_out

    loss = torch.linalg.norm(k_data_pred - k_data_tar, ord=1) / torch.linalg.norm(
        k_data_tar, ord=1
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    progress_bar.set_postfix(loss=f"{loss.item():.4f}")

plt.figure()
plt.plot(losses, marker="o")
plt.xlabel("Split")
plt.ylabel("Self-supervised loss")
plt.title("DeepADMM training loss over splits")
plt.show()

## Task 4: Inference on Trained Model
 
With `rho` and the regularizer's weights now trained, evaluate the model exactly as in Task 2 — but on the **full, unsplit** k-space, since at inference time you want the best reconstruction using every sample you acquired.
 
**Your tasks:**
 
1. Call `model.eval()` to put the trained network in evaluation mode.
2. Reuse `Q`/`b` from the CG reconstruction above, already computed from the full trajectory.
3. Run `model(Q, b, b)` inside a `torch.no_grad()` block and reshape the result for the final comparison.

In [ ]:
# Your solution here




## Comparison

You now have four reconstructions of the same undersampled radial acquisition: CG (unregularized), classical ADMM (fixed TV prior), DeepADMM with random weights, and DeepADMM after self-supervised training. Compare them and answer the following:
 
1. **Untrained vs. trained DeepADMM.** How different are the two DeepADMM reconstructions? Is the untrained network doing anything useful on its own, or is essentially all of the image quality coming from training? What does this tell you about how much of the improvement is due to the unrolled ADMM *structure* versus the *learned weights*?
2. **Trained DeepADMM vs. classical ADMM.** Which reconstruction looks sharper, and which suppresses streaking artifacts (from undersampling) more effectively? Are there new artifacts in the learned reconstruction that weren't present in either the CG or classical ADMM images?
3. **No ground truth, no external data.** The network was never shown a fully-sampled reference image, nor any other patient's scan — it only ever saw random halves of this one acquisition's own k-space. Given that, does it surprise you that this works at all? What in the loss function is actually driving the network toward better images, if not a direct image-domain comparison to ground truth?
4. **Effect of `R`.** Go back to the **System and Sequence Parameters** cell and try a smaller and a larger acceleration factor `R`. Does the *gap* between classical ADMM and trained DeepADMM grow or shrink as undersampling increases? Where would you expect a learned prior to offer the biggest advantage over a fixed one, and why?
5. **Limitations of this setup.** `num_splits` controls how many self-supervised training steps are taken — all on a *single* scan. What would you expect to happen to the trained model's performance on a *different* phantom or a different anatomical slice that it never saw during training? What would you need to change about this training loop to get a model that generalizes across scans rather than one specialized to this single acquisition?

In [ ]:
plt.figure(figsize=(10, 10))
plt.subplot(2, 2, 1)
plt.imshow(torch.abs(x_recon_image).numpy().T, cmap="gray")
plt.title("CG")
plt.subplot(2, 2, 2)
plt.imshow(torch.abs(x_admm_image).numpy().T, cmap="gray")
plt.title("Classic ADMM (fixed TV)")
plt.subplot(2, 2, 3)
plt.imshow(torch.abs(x_deep_admm_init_image).numpy().T, cmap="gray")
plt.title("DeepADMM (learned, not trained)")
plt.subplot(2, 2, 4)
plt.imshow(torch.abs(x_deep_admm_trained_image).numpy().T, cmap="gray")
plt.title("DeepADMM (learned, trained)")
plt.show()